# Stage 4: HDBSCAN Clustering and Review Diagnostics

## Overview and Purpose
Grouping anime characters without prior labels presents unique unsupervised clustering challenges:
- The total number of distinct characters $K$ is unknown beforehand.
- Cluster sizes are highly unbalanced (main characters may appear in hundreds of images, while guest characters appear in two or three).
- Non-character images, landscapes, or low-quality crops must not be forced into character clusters.

This notebook executes **Stage 4** using **HDBSCAN** (Hierarchical Density-Based Spatial Clustering of Applications with Noise) optimized for anime character embeddings:
1. **Optimized Parameters**:
   - `min_cluster_size = 2`: Allows small character groups (pairs and triplets) to form valid clusters rather than being discarded as noise.
   - `min_samples = 1`: Eliminates severe reachability distance penalties on sparse/small groups.
   - `cluster_selection_epsilon = 0.50`: Merges sub-clusters within CCIP visual character identity threshold (~0.50).
2. **Cluster Centroids**: Calculates $L_2$-normalized centroids for each cluster on the unit hypersphere and computes Euclidean distance metrics.
3. **Soft Noise Reassignment**: Recovers borderline noise points that lie within `max_reassign_distance = 0.55` of an established cluster centroid.
4. **Diagnostic Metrics**: Calculates cluster membership probabilities, GLOSH outlier scores, and flags uncertain items for review.
5. **Visual Diagnostics**: Generates cluster size distributions, outlier plots, and representative contact sheets.

## 1. Imports and Environment Setup
Import clustering routines, diagnostics, and visualization helpers.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

from anime_character_organizer import (
    ClusteringConfig,
    run_clustering,
)
from anime_character_organizer.clustering.algorithm import fit_hdbscan
from anime_character_organizer.clustering.centroids import compute_cluster_centroids
from anime_character_organizer.visualization.plots import (
    plot_cluster_size_distribution,
    plot_probability_and_outlier_distributions,
)
from anime_character_organizer.utils.runs import find_latest_run

## 2. Clustering Configuration & Hyperparameter Optimization
Specify HDBSCAN parameters and noise reassignment settings.

In [ ]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_EMBED_RUN = None  # Auto-discovers latest 03_ccip_embeddings run

# HDBSCAN parameters optimized for character identity clustering:
MIN_CLUSTER_SIZE = 2              # Allows pairs and triplets of characters to form clusters
MIN_SAMPLES = 1                   # Minimizes reachability distance penalty on small clusters
CLUSTER_SELECTION_EPSILON = 0.50   # Merges points within close visual distance
CLUSTER_SELECTION_METHOD = "eom"  # Excess of Mass
METRIC = "euclidean"

# Soft noise reassignment: recovers borderline noise items within threshold of a cluster centroid
REASSIGN_NOISE = True
MAX_REASSIGN_DISTANCE = 0.55

# Diagnostic review thresholds
LOW_PROBABILITY_THRESHOLD = 0.35
HIGH_OUTLIER_QUANTILE = 0.95

# Keep character folders unified (False) rather than splitting into separate _review folders
SEPARATE_REVIEW_FOLDERS = False

print("Clustering Configuration:")
print(f"  min_cluster_size={MIN_CLUSTER_SIZE}, min_samples={MIN_SAMPLES}, epsilon={CLUSTER_SELECTION_EPSILON}")
print(f"  Noise Reassignment: enabled={REASSIGN_NOISE}, max_distance={MAX_REASSIGN_DISTANCE}")
print(f"  Separate Review Folders: {SEPARATE_REVIEW_FOLDERS}")

## 3. HDBSCAN Clustering Workflow Execution
Run the clustering workflow. This extracts centroids, computes distances, reassigns borderline noise, and generates diagnostic plots.

In [ ]:
results = run_clustering(
    project_dir=PROJECT_DIR,
    previous_run_dir=PREVIOUS_EMBED_RUN,
    min_cluster_size=MIN_CLUSTER_SIZE,
    min_samples=MIN_SAMPLES,
    cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
    cluster_selection_method=CLUSTER_SELECTION_METHOD,
    metric=METRIC,
    low_probability_threshold=LOW_PROBABILITY_THRESHOLD,
    high_outlier_quantile=HIGH_OUTLIER_QUANTILE,
    reassign_noise=REASSIGN_NOISE,
    max_reassign_distance=MAX_REASSIGN_DISTANCE,
    separate_review_folders=SEPARATE_REVIEW_FOLDERS,
    show_progress=True,
)

print(f"Clustering completed.")
print(f"Run directory: {results['run_dir']}")

## 4. Cluster Summary & Diagnostic Metrics
Inspect aggregated statistics for each discovered cluster (image count, mean probability, outlier score, centroid distance).

In [ ]:
summary = results["summary"]
print(f"Total Embeddings:       {summary['embedding_count']}")
print(f"Estimated Clusters:     {summary['estimated_clusters']}")
print(f"Clustered Images:       {summary['clustered_points']}")
print(f"Remaining Noise Items:  {summary['noise_points']}")
print(f"Reassigned from Noise:  {summary['reassigned_points']}")
print(f"Flagged for Review:     {summary['manual_review_points']}")

cluster_summary_df = results["cluster_summary_df"]
display(cluster_summary_df[[
    "cluster_label", "cluster_folder_stub", "is_noise_cluster",
    "image_count", "requires_review_count", "requires_review_share",
    "mean_probability", "mean_distance_to_centroid"
]].head(30))

## 5. Visual Diagnostics: Cluster Size & Probability Distributions
Inspect diagnostic plots illustrating cluster size distributions and probability/outlier densities.

In [ ]:
fig1 = plot_cluster_size_distribution(cluster_summary_df)
plt.show()

fig2 = plot_probability_and_outlier_distributions(
    results["cluster_manifest_df"],
    low_probability_threshold=LOW_PROBABILITY_THRESHOLD,
    high_outlier_threshold=summary["high_outlier_threshold"],
)
plt.show()

## 6. Folder Assignment Plan Preview
Preview how images will be assigned to proposed destination folder stubs (`cluster_XXXXX_unknown` or `_needs_review_noise`).

In [ ]:
folder_assignment_df = results["folder_assignment_df"]
display(folder_assignment_df[[
    "embedding_row", "relative_path", "cluster_label",
    "cluster_probability", "outlier_score", "distance_to_centroid",
    "requires_manual_review", "proposed_folder"
]].head(30))

## 7. Assumptions, Limitations, and Next Steps

### Assumptions & Limitations
- **Conservative Clustering Strategy**: The pipeline prefers producing small, clean clusters over aggressive merging. It is significantly easier for a user to merge two folders of the same character than to separate a contaminated folder containing two distinct characters.
- **Twin / Recolor Characters**: Characters sharing identical face and hair geometries (e.g. palette swaps, identical twins) will cluster together based on visual embeddings. Stage 6 tagger naming helps disambiguate these cases.

### Next Steps
The folder assignment manifest `folder_assignment_manifest.csv` is passed to **Stage 5 (`05_non_destructive_folder_materialization.ipynb`)** to materialize the physical folder structure.